# Train LED on 1,200 legal cases

This notebook teaches LED to summarize judgments using their existing summaries. Enable a GPU in Kaggle before running.

## Install packages
Installs the libraries this notebook needs.

In [ ]:
%pip install -q "transformers>=4.45,<5" sentencepiece accelerate

## Set up training
Loads the tools, checks the GPU, and sets the training options.

In [1]:
import json, random, time
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, LEDForConditionalGeneration

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not active. Select an accelerator in Kaggle Settings.")

MODEL_NAME = "allenai/led-base-16384"
MAX_SOURCE_LENGTH = 4096
MAX_TARGET_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
EPOCHS = 3
LEARNING_RATE = 2e-5
OUTPUT_DIR = Path("/kaggle/working/led_legal_untagged_1200")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

I0000 00:00:1794677605.823065     134 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Device: cuda
GPU: Tesla T4


## Match judgments and summaries
Pairs each judgment with its summary using the filenames and selects up to 1,200 cases.

In [2]:
from pathlib import Path

JUDGMENT_DIR = Path("/kaggle/input/datasets/aizennnnnn/dataset/train/judgement")
SUMMARY_DIR = Path("/kaggle/input/datasets/aizennnnnn/dataset/train/summary")

if not JUDGMENT_DIR.is_dir():
    raise FileNotFoundError(f"Judgments folder not found: {JUDGMENT_DIR}")
if not SUMMARY_DIR.is_dir():
    raise FileNotFoundError(f"Summaries folder not found: {SUMMARY_DIR}")

def sort_key(path):
    return (0, int(path.stem)) if path.stem.isdigit() else (1, path.stem.lower())

summary_files = {p.stem: p for p in SUMMARY_DIR.glob("*.txt")}
judgment_files = sorted(JUDGMENT_DIR.glob("*.txt"), key=sort_key)

pairs = []
missing_summaries = []

for judgment_path in judgment_files:
    summary_path = summary_files.get(judgment_path.stem)
    if summary_path is None:
        missing_summaries.append(judgment_path.name)
        continue

    document = judgment_path.read_text(encoding="utf-8", errors="replace").strip()
    summary = summary_path.read_text(encoding="utf-8", errors="replace").strip()

    if document and summary:
        pairs.append({
            "id": judgment_path.stem,
            "document": document,
            "summary": summary,
        })

    if len(pairs) % 200 == 0 and pairs:
        print(f"Loaded {len(pairs)} pairs")

    if len(pairs) == 1200:
        break

if len(pairs) < 2:
    raise ValueError(f"Found only {len(pairs)} matched non-empty pairs.")

print("Matched pairs loaded:", len(pairs))
print("Missing summaries encountered:", len(missing_summaries))
print("Example case ID:", pairs[0]["id"])
print("Document characters:", len(pairs[0]["document"]))
print("Summary characters:", len(pairs[0]["summary"]))

Loaded 200 pairs
Loaded 400 pairs
Loaded 600 pairs
Loaded 800 pairs
Loaded 1000 pairs
Loaded 1200 pairs
Matched pairs loaded: 1200
Missing summaries encountered: 0
Example case ID: 1
Document characters: 18758
Summary characters: 836


## Prepare training and validation sets
Sets aside some cases for checking how well the model summarizes.

In [6]:
random.Random(SEED).shuffle(pairs)
n_val = max(1, round(len(pairs) * 0.10))
validation_examples = pairs[:n_val]
train_examples = pairs[n_val:]
print("Train:", len(train_examples), "| validation:", len(validation_examples))

Train: 1080 | validation: 120


## Load LED
Loads the pretrained model that will learn to summarize legal judgments.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = LEDForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.use_cache = False
if DEVICE.type == "cuda":
    model.gradient_checkpointing_enable()
model.to(DEVICE)

ATTENTION_WINDOW = model.config.attention_window
if isinstance(ATTENTION_WINDOW, (list, tuple)):
    ATTENTION_WINDOW = max(ATTENTION_WINDOW)
print("LED ready. Attention window:", ATTENTION_WINDOW)

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/648M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

LED ready. Attention window: 1024


## Prepare batches
Turns the text into model inputs for training and validation.

In [8]:
class SummaryDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        row = self.rows[i]
        x = tokenizer(row["document"], max_length=MAX_SOURCE_LENGTH, truncation=True)
        y = tokenizer(text_target=row["summary"], max_length=MAX_TARGET_LENGTH, truncation=True)
        return {"input_ids": x["input_ids"], "attention_mask": x["attention_mask"], "labels": y["input_ids"]}

def collate_batch(items):
    source = tokenizer.pad(
        [{"input_ids": x["input_ids"], "attention_mask": x["attention_mask"]} for x in items],
        padding=True, pad_to_multiple_of=ATTENTION_WINDOW, return_tensors="pt"
    )
    target = tokenizer.pad(
        [{"input_ids": x["labels"]} for x in items], padding=True, return_tensors="pt"
    )["input_ids"]
    target[target == tokenizer.pad_token_id] = -100
    global_mask = torch.zeros_like(source["input_ids"])
    global_mask[:, 0] = 1
    return {"input_ids": source["input_ids"], "attention_mask": source["attention_mask"],
            "global_attention_mask": global_mask, "labels": target}

train_loader = DataLoader(SummaryDataset(train_examples), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
validation_loader = DataLoader(SummaryDataset(validation_examples), batch_size=1, shuffle=False, collate_fn=collate_batch)
print("Train batches:", len(train_loader), "| validation batches:", len(validation_loader))

Train batches: 1080 | validation batches: 120


## 6. Train Longformer Encoder Decoder Using Rhetorical Tagged judgements
The model architecture used in this project is *Longformer-Encoder-Decoder (LED), introduced by Beltagy, Peters, and Cohan in *Longformer: The Long-Document Transformer (2020). LED adapts Longformer for long-document generation by combining an encoder with efficient local and global attention and a text-generating decoder. We use the pretrained allenai/led-base-16384 checkpoint through Hugging Face Transformers and fine-tune it on our rhetorically tagged legal judgments using LoRA.

*Reference:* Beltagy, I., Peters, M. E., & Cohan, A. (2020). [Longformer: The Long-Document Transformer](https://arxiv.org/abs/2004.05150).

In [9]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
history = []

def to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_sum = 0.0
    started = time.time()

    for step, batch in enumerate(train_loader, 1):
        batch = to_device(batch)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            loss = model(**batch).loss
        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        train_sum += loss.detach().item()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        if step % 25 == 0 or step == len(train_loader):
            print(f"Epoch {epoch}/{EPOCHS} | batch {step}/{len(train_loader)} | mean train loss {train_sum / step:.4f}")

    train_loss = train_sum / max(1, len(train_loader))
    model.eval()
    val_sum = 0.0
    with torch.no_grad():
        for batch in validation_loader:
            batch = to_device(batch)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                val_sum += model(**batch).loss.item()

    val_loss = val_sum / max(1, len(validation_loader))
    history.append({"epoch": epoch, "train_loss": train_loss, "validation_loss": val_loss})
    print(f"Epoch {epoch} done | train loss {train_loss:.4f} | validation loss {val_loss:.4f} | {(time.time()-started)/60:.1f} min")

    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    (OUTPUT_DIR / "training_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    print("Checkpoint saved:", OUTPUT_DIR)

print("Training complete:", OUTPUT_DIR)

You're using a LEDTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch 1/3 | batch 25/1080 | mean train loss 3.2348
Epoch 1/3 | batch 50/1080 | mean train loss 2.9696
Epoch 1/3 | batch 75/1080 | mean train loss 2.8912
Epoch 1/3 | batch 100/1080 | mean train loss 2.8520
Epoch 1/3 | batch 125/1080 | mean train loss 2.7670
Epoch 1/3 | batch 150/1080 | mean train loss 2.6882
Epoch 1/3 | batch 175/1080 | mean train loss 2.6351
Epoch 1/3 | batch 200/1080 | mean train loss 2.5982
Epoch 1/3 | batch 225/1080 | mean train loss 2.5463
Epoch 1/3 | batch 250/1080 | mean train loss 2.5340
Epoch 1/3 | batch 275/1080 | mean train loss 2.5273
Epoch 1/3 | batch 300/1080 | mean train loss 2.5016
Epoch 1/3 | batch 325/1080 | mean train loss 2.4804
Epoch 1/3 | batch 350/1080 | mean train loss 2.4632
Epoch 1/3 | batch 375/1080 | mean train loss 2.4466
Epoch 1/3 | batch 400/1080 | mean train loss 2.4356
Epoch 1/3 | batch 425/1080 | mean train loss 2.4228
Epoch 1/3 | batch 450/1080 | mean train loss 2.4160
Epoch 1/3 | batch 475/1080 | mean train loss 2.4052
Epoch 1/3 | bat

In [13]:
%pip install -q rouge-score bert-score

Note: you may need to restart the kernel to use updated packages.


In [14]:
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/aizennnnnn/dataset")

test_roots = [
    DATASET_ROOT / "test",
    DATASET_ROOT / "test-data",
]

for test_root in test_roots:
    test_judgment_dir = test_root / "judgement"
    test_summary_dir = test_root / "summary"
    if test_judgment_dir.is_dir() and test_summary_dir.is_dir():
        break
else:
    raise FileNotFoundError(
        f"Could not find test/judgement and test/summary under {DATASET_ROOT}"
    )

test_summaries = {p.stem: p for p in test_summary_dir.glob("*.txt")}
test_examples = []

for judgment_path in sorted(
    test_judgment_dir.glob("*.txt"),
    key=lambda p: (0, int(p.stem)) if p.stem.isdigit() else (1, p.stem.lower()),
):
    summary_path = test_summaries.get(judgment_path.stem)
    if summary_path is None:
        continue

    document = judgment_path.read_text(encoding="utf-8", errors="replace").strip()
    summary = summary_path.read_text(encoding="utf-8", errors="replace").strip()

    if document and summary:
        test_examples.append({
            "id": judgment_path.stem,
            "document": document,
            "summary": summary,
        })

print("Matched test pairs:", len(test_examples))
if len(test_examples) != 100:
    print("Check the test folders: expected 100 matched pairs.")

Matched test pairs: 100


In [15]:
import torch
import pandas as pd
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from transformers import AutoTokenizer, LEDForConditionalGeneration
from pathlib import Path

MODEL_DIR = Path("/kaggle/working/led_legal_untagged_1200")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not MODEL_DIR.is_dir():
    raise FileNotFoundError(f"Fine-tuned model not found: {MODEL_DIR}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = LEDForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

attention_window = model.config.attention_window
if isinstance(attention_window, (list, tuple)):
    attention_window = max(attention_window)

predictions = []
references = []
case_ids = []

for i, example in enumerate(test_examples, start=1):
    inputs = tokenizer(
        example["document"],
        return_tensors="pt",
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=True,
        pad_to_multiple_of=attention_window,
    )
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    global_mask = torch.zeros_like(inputs["input_ids"])
    global_mask[:, 0] = 1

    with torch.no_grad():
        generated = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            global_attention_mask=global_mask,
            max_new_tokens=350,
            num_beams=2,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    predictions.append(tokenizer.decode(generated[0], skip_special_tokens=True))
    references.append(example["summary"])
    case_ids.append(example["id"])

    if i % 5 == 0 or i == len(test_examples):
        print(f"Generated {i}/{len(test_examples)} test summaries")

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)

per_case = []
for case_id, prediction, reference in zip(case_ids, predictions, references):
    scores = rouge.score(reference, prediction)
    per_case.append({
        "case_id": case_id,
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
        "reference": reference,
        "generated": prediction,
    })

model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

precision, recall, f1 = bert_score(
    predictions,
    references,
    model_type="roberta-large",
    lang="en",
    device=str(DEVICE),
    batch_size=4,
    verbose=True,
)

for i, row in enumerate(per_case):
    row["bertscore_precision"] = precision[i].item()
    row["bertscore_recall"] = recall[i].item()
    row["bertscore_f1"] = f1[i].item()

per_case_df = pd.DataFrame(per_case)
metric_columns = [
    "rouge1", "rouge2", "rougeL",
    "bertscore_precision", "bertscore_recall", "bertscore_f1",
]

metrics_matrix = pd.DataFrame([{
    "Model": "LED untagged",
    "Test judgments": len(per_case_df),
    **{metric: per_case_df[metric].mean() for metric in metric_columns},
}])

display(metrics_matrix.round(4))
display(per_case_df[["case_id"] + metric_columns].round(4).head(10))

per_case_df.to_csv(
    "/kaggle/working/led_untagged_test_100_predictions_and_scores.csv",
    index=False,
)
metrics_matrix.to_csv(
    "/kaggle/working/led_untagged_test_100_metrics_matrix.csv",
    index=False,
)

print("Saved per-case results and metrics matrix in /kaggle/working")

Generated 5/100 test summaries
Generated 10/100 test summaries
Generated 15/100 test summaries
Generated 20/100 test summaries
Generated 25/100 test summaries
Generated 30/100 test summaries
Generated 35/100 test summaries
Generated 40/100 test summaries
Generated 45/100 test summaries
Generated 50/100 test summaries
Generated 55/100 test summaries
Generated 60/100 test summaries
Generated 65/100 test summaries
Generated 70/100 test summaries
Generated 75/100 test summaries
Generated 80/100 test summaries
Generated 85/100 test summaries
Generated 90/100 test summaries
Generated 95/100 test summaries
Generated 100/100 test summaries


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

done in 18.32 seconds, 5.46 sentences/sec


,Model,Test judgments,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1
0,LED untagged,100,0.328,0.1664,0.192,0.8608,0.8303,0.8451


,case_id,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1
0,78,0.5761,0.2607,0.2393,0.8587,0.8489,0.8538
1,232,0.1494,0.0919,0.1084,0.8714,0.8151,0.8423
2,266,0.4196,0.1677,0.2177,0.8442,0.8378,0.8410
3,314,0.4118,0.1591,0.2398,0.8442,0.8443,0.8443
4,362,0.3366,0.1844,0.1826,0.8576,0.8492,0.8534
5,380,0.5972,0.3369,0.2898,0.8683,0.8507,0.8594
6,415,0.2642,0.0758,0.1651,0.8448,0.8094,0.8267
7,496,0.3366,0.0526,0.1655,0.8149,0.7968,0.8057
8,652,0.4672,0.1705,0.2355,0.8285,0.8433,0.8359
9,660,0.4293,0.1393,0.2010,0.8299,0.8081,0.8189


Saved per-case results and metrics matrix in /kaggle/working


In [16]:
import json
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/led_legal_untagged_1200/test_evaluation_100")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

metrics = metrics_matrix.iloc[0].to_dict()

with (SAVE_DIR / "test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, default=str)

metrics_matrix.to_csv(SAVE_DIR / "test_metrics_matrix.csv", index=False)
per_case_df.to_csv(SAVE_DIR / "per_case_predictions_and_scores.csv", index=False)

print("Saved test metrics and per-case results to:", SAVE_DIR)

Saved test metrics and per-case results to: /kaggle/working/led_legal_untagged_1200/test_evaluation_100


In [18]:
import csv
import json

with (SAVE_DIR / "test_split_manifest.csv").open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.writer(f)
    writer.writerow(["id", "split"])
    for example in test_examples:
        writer.writerow([example["id"], "test"])

details = {
    "model": "LED fine-tuned on untagged judgments",
    "test_judgments": len(test_examples),
    "source_max_tokens": MAX_SOURCE_LENGTH,
    "metrics": metrics,
}

with (SAVE_DIR / "evaluation_details.json").open("w", encoding="utf-8") as f:
    json.dump(details, f, indent=2, default=str)

print("Evaluation files:", sorted(path.name for path in SAVE_DIR.iterdir()))

Evaluation files: ['evaluation_details.json', 'per_case_predictions_and_scores.csv', 'test_metrics.json', 'test_metrics_matrix.csv', 'test_split_manifest.csv']
